# 🏆 Akkadian-English Translation: Inference Notebook

This notebook performs inference using a trained ByT5 model.

**Key Post-Processing:**
- Gap normalization (x → `<gap>`, ... → `<big_gap>`)
- Decimal to fraction conversion (0.5 → ½)
- Task prefix for model input

**Usage:**
1. Add your trained model as a Kaggle dataset
2. Update `MODEL_PATH` below
3. Run all cells

In [1]:
"""
================================================================================
STUDENT MODEL TRAINING - KNOWLEDGE DISTILLATION
================================================================================
This notebook trains a single ByT5-base model to match the 34.3 LB teacher ensemble.

REQUIREMENTS:
- Teacher translations CSV (from translate_train_csv.py)
- Kaggle GPU (P100/T4)
- ~4-6 hours training time

CRITICAL SETTINGS:
- FP32 ONLY (no bf16/fp16) - prevents hallucination
- Same preprocessing as teacher
- Same generation params as 34.3 baseline
================================================================================
"""

# ============================================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================================
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate

import os
import re
import gc
import torch
import numpy as np
import pandas as pd
from datetime import datetime

# Clear memory
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("STUDENT MODEL TRAINING")
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================

class Config:
    """All training configuration in one place."""
    
    # Paths - adjust for your setup
    TEACHER_CSV = "/kaggle/input/ewsetd/train_with_teacher_translations.csv"
    # Alternative: use original train.csv with ground truth
    TRAIN_CSV = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
    
    # Model
    MODEL_NAME = "google/byt5-base"
    OUTPUT_DIR = "./student_model"
    
    # Training hyperparameters - MEMORY OPTIMIZED!
    NUM_EPOCHS = 15
    BATCH_SIZE = 2  # Reduced from 4
    GRADIENT_ACCUMULATION = 8  # Increased to maintain effective batch = 16
    LEARNING_RATE = 5e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # Generation (same as 34.3 teacher!)
    MAX_INPUT_LENGTH = 256  # Reduced from 512
    MAX_TARGET_LENGTH = 256  # Reduced from 512
    NUM_BEAMS = 10
    LENGTH_PENALTY = 1.15
    
    # CRITICAL: FP32 ONLY!
    USE_FP16 = False
    USE_BF16 = False
    
    # Data strategy
    USE_TEACHER_TRANSLATIONS = True  # If False, use ground truth only
    
    # Prefix (same as teacher)
    PREFIX = "translate Akkadian to English: "
    
    # Evaluation
    EVAL_STEPS = 200  # Less frequent eval to save time
    SAVE_STEPS = 200
    LOGGING_STEPS = 50


config = Config()

print(f"\n📋 Configuration:")
print(f"   Model: {config.MODEL_NAME}")
print(f"   Epochs: {config.NUM_EPOCHS}")
print(f"   Batch Size: {config.BATCH_SIZE} x {config.GRADIENT_ACCUMULATION} = {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")
print(f"   Learning Rate: {config.LEARNING_RATE}")
print(f"   FP16: {config.USE_FP16}, BF16: {config.USE_BF16}")
print(f"   Use Teacher Translations: {config.USE_TEACHER_TRANSLATIONS}")

# ============================================================================
# CELL 3: PREPROCESSING FUNCTIONS (Same as 34.3!)
# ============================================================================

def replace_gaps(text):
    """Standardize gap patterns in input."""
    if not isinstance(text, str):
        return ""
    
    # Standardize various gap patterns
    text = re.sub(r'\[\s*\.\s*\.\s*\.\s*\]', '...', text)
    text = re.sub(r'\(\s*\.\s*\.\s*\.\s*\)', '...', text)
    text = re.sub(r'\.{3,}', '...', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def preprocess_input(text):
    """Preprocess transliteration input."""
    text = replace_gaps(text)
    return config.PREFIX + text


def postprocess_output(text):
    """Post-process model output (same as 34.3!)."""
    if not isinstance(text, str):
        return ""
    
    # 1. ḫ/Ḫ → h/H conversion
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    
    # 2. Subscript normalization
    subscripts = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')
    text = text.translate(subscripts)
    
    # 3. Superscript normalization
    superscripts = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹', '0123456789')
    text = text.translate(superscripts)
    
    # 4. Fraction conversion
    fractions = {
        '1/2': '½', '1/3': '⅓', '2/3': '⅔',
        '1/4': '¼', '3/4': '¾', '1/5': '⅕',
        '1/6': '⅙', '5/6': '⅚', '1/8': '⅛',
    }
    for frac, symbol in fractions.items():
        text = text.replace(frac, symbol)
    
    # 5. Clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


print("✓ Preprocessing functions defined")

# ============================================================================
# CELL 4: LOAD DATA
# ============================================================================

print("\n📂 Loading data...")

# Try to load teacher translations first
if config.USE_TEACHER_TRANSLATIONS and os.path.exists(config.TEACHER_CSV):
    df = pd.read_csv(config.TEACHER_CSV)
    print(f"   Loaded teacher translations: {len(df)} rows")
    
    # Use teacher translations as target
    if 'teacher_translation' in df.columns:
        df['target'] = df['teacher_translation']
        print("   Using: teacher_translation as target")
    else:
        df['target'] = df['translation']
        print("   Warning: No teacher_translation column, using ground truth")
else:
    # Fall back to original train.csv
    df = pd.read_csv(config.TRAIN_CSV)
    df['target'] = df['translation']
    print(f"   Loaded ground truth: {len(df)} rows")
    print("   Using: ground truth translation as target")

# Preprocess
df['input_text'] = df['transliteration'].apply(preprocess_input)
df['target_text'] = df['target'].apply(lambda x: str(x) if pd.notna(x) else "")

# Remove any empty rows
df = df[df['target_text'].str.len() > 0]
print(f"   After cleaning: {len(df)} rows")

# Show sample
print("\n📝 Sample data:")
print(f"   Input:  {df['input_text'].iloc[0][:80]}...")
print(f"   Target: {df['target_text'].iloc[0][:80]}...")

# ============================================================================
# CELL 5: CREATE DATASET
# ============================================================================

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

print("\n🔤 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Split data (90% train, 10% eval)
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"   Train: {len(train_df)}, Eval: {len(eval_df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['input_text', 'target_text']].reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df[['input_text', 'target_text']].reset_index(drop=True))


def tokenize_function(examples):
    """Tokenize inputs and targets for ByT5."""
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=config.MAX_INPUT_LENGTH,
        padding='max_length',
        truncation=True,
    )
    
    # Tokenize targets
    labels = tokenizer(
        examples['target_text'],
        max_length=config.MAX_TARGET_LENGTH,
        padding='max_length',
        truncation=True,
    )
    
    # Replace padding token id with -100 so it's ignored in loss
    label_ids = labels['input_ids']
    # Convert to list of lists for processing
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in label_ids
    ]
    
    model_inputs['labels'] = label_ids
    return model_inputs


print("   Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])
eval_dataset = eval_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])

print(f"   ✓ Train dataset: {len(train_dataset)} samples")
print(f"   ✓ Eval dataset: {len(eval_dataset)} samples")

# ============================================================================
# CELL 6: LOAD MODEL
# ============================================================================

from transformers import T5ForConditionalGeneration

print("\n🤖 Loading model...")

# Load model - DO NOT call .float() as it can break gradients!
model = T5ForConditionalGeneration.from_pretrained(
    config.MODEL_NAME,
    torch_dtype=torch.float32,  # Use float32 via config instead
)

# Ensure model is in training mode
model.train()

print(f"   Model dtype: {next(model.parameters()).dtype}")
print(f"   Training mode: {model.training}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total params: {total_params:,}")
print(f"   Trainable: {trainable_params:,}")

# ============================================================================
# CELL 7: EVALUATION METRICS
# ============================================================================

import evaluate

# Load metrics
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    """Compute BLEU and chrF++ on evaluation set."""
    predictions, labels = eval_preds
    
    # Replace -100 and invalid IDs in predictions with pad token
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    predictions = np.where(predictions >= tokenizer.vocab_size, tokenizer.pad_token_id, predictions)
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Decode labels (replace -100 with pad token)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    labels = np.where(labels >= tokenizer.vocab_size, tokenizer.pad_token_id, labels)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Apply post-processing (same as 34.3!)
    decoded_preds = [postprocess_output(p) for p in decoded_preds]
    decoded_labels = [postprocess_output(l) for l in decoded_labels]
    
    # Handle empty predictions
    decoded_preds = [p if p.strip() else "." for p in decoded_preds]
    decoded_labels = [l if l.strip() else "." for l in decoded_labels]
    
    # Compute BLEU
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )
    
    # Compute chrF++
    chrf_result = chrf_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
        word_order=2
    )
    
    return {
        "bleu": bleu_result["score"],
        "chrf": chrf_result["score"],
    }


print("✓ Metrics defined (BLEU, chrF++)")

# ============================================================================
# CELL 8: TRAINING SETUP
# ============================================================================

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=config.OUTPUT_DIR,
    
    # Training schedule
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,
    
    # Learning rate
    learning_rate=config.LEARNING_RATE,
    warmup_ratio=config.WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=config.WEIGHT_DECAY,
    
    # CRITICAL: FP32 ONLY!
    bf16=config.USE_BF16,
    fp16=config.USE_FP16,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=config.EVAL_STEPS,
    
    # Saving - ONLY KEEP BEST MODEL
    save_strategy="steps",
    save_steps=config.SAVE_STEPS,
    save_total_limit=1,  # Only keep the BEST model
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    
    # Logging
    logging_steps=config.LOGGING_STEPS,
    logging_dir="./logs",
    report_to="none",  # Disable wandb etc
    
    # Generation for eval
    predict_with_generate=True,
    generation_max_length=config.MAX_TARGET_LENGTH,
    generation_num_beams=4,  # Smaller for speed during eval
    
    # Other
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    gradient_checkpointing=True,  # ENABLED for memory savings
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n⚙️ Training configuration:")
print(f"   Epochs: {config.NUM_EPOCHS}")
print(f"   Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")
print(f"   Learning rate: {config.LEARNING_RATE}")
print(f"   FP32 training: {not (config.USE_FP16 or config.USE_BF16)}")

# ============================================================================
# CELL 9: TRAIN!
# ============================================================================

print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)

# AGGRESSIVE DISK CLEANUP before training
import shutil

def clear_disk_space():
    """Clear disk space aggressively."""
    import subprocess
    
    # Clear old checkpoints
    if os.path.exists(config.OUTPUT_DIR):
        shutil.rmtree(config.OUTPUT_DIR, ignore_errors=True)
        print("   Cleared old checkpoints")
    
    # Clear HuggingFace cache
    cache_dirs = [
        "/root/.cache/huggingface/hub",
        "/kaggle/temp",
        "./logs",
    ]
    for cache_dir in cache_dirs:
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir, ignore_errors=True)
    
    # Clear Python cache
    gc.collect()
    torch.cuda.empty_cache()
    
    # Show disk usage
    try:
        result = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True)
        print(f"   Disk usage after cleanup:\n{result.stdout}")
    except:
        pass

clear_disk_space()

# Custom callback to clear disk before each save
from transformers import TrainerCallback

# Load a few test samples for demo translations
TEST_CSV = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
test_samples_df = pd.read_csv(TEST_CSV)
DEMO_SAMPLES = test_samples_df['transliteration'].head(3).tolist()  # 3 samples for demo

class DiskSpaceCallback(TrainerCallback):
    """Callback to manage disk space and show demo translations."""
    
    def __init__(self, output_dir, model, tokenizer, demo_samples):
        self.output_dir = output_dir
        self.model = model
        self.tokenizer = tokenizer
        self.demo_samples = demo_samples
        self.best_metric = -float('inf')
        self.best_checkpoint = None
    
    def on_save(self, args, state, control, **kwargs):
        """Clear old checkpoints before saving new one."""
        # Find and delete old checkpoints to free space BEFORE saving
        if os.path.exists(self.output_dir):
            for item in os.listdir(self.output_dir):
                item_path = os.path.join(self.output_dir, item)
                if item.startswith("checkpoint-") and os.path.isdir(item_path):
                    # Delete all old checkpoints (keep only current)
                    if item_path != self.best_checkpoint:
                        shutil.rmtree(item_path, ignore_errors=True)
                        print(f"   🗑️ Deleted old checkpoint: {item}")
        
        # Delete optimizer.pt to save 1GB
        optimizer_files = ["optimizer.pt", "rng_state.pth"]
        for opt_file in optimizer_files:
            opt_path = os.path.join(self.output_dir, opt_file)
            if os.path.exists(opt_path):
                os.remove(opt_path)
        
        # Clear cache
        gc.collect()
        torch.cuda.empty_cache()
        return control
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """Track best checkpoint and show demo translations."""
        if metrics:
            current_metric = metrics.get("eval_bleu", 0)
            
            # Print demo translations
            print(f"\n{'─'*60}")
            print(f"📝 DEMO TRANSLATIONS (Step {state.global_step}, BLEU: {current_metric:.2f})")
            print(f"{'─'*60}")
            
            self.model.eval()
            device = next(self.model.parameters()).device
            
            for i, sample in enumerate(self.demo_samples[:2]):  # Show 2 samples
                input_text = preprocess_input(sample)
                inputs = self.tokenizer(
                    input_text, 
                    return_tensors="pt", 
                    padding=True, 
                    truncation=True,
                    max_length=256
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=256,
                        num_beams=4,  # Faster for demo
                        length_penalty=1.15,
                        early_stopping=True,
                    )
                
                decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                decoded = postprocess_output(decoded)
                
                print(f"\n[{i+1}] Input:  {sample[:60]}...")
                print(f"    Output: {decoded[:80]}...")
            
            print(f"{'─'*60}\n")
            
            # Track best
            if current_metric > self.best_metric:
                self.best_metric = current_metric
                self.best_checkpoint = os.path.join(
                    self.output_dir, 
                    f"checkpoint-{state.global_step}"
                )
                print(f"   ⭐ New best BLEU: {current_metric:.2f}")
            
            self.model.train()
        
        return control

# Add the callback
trainer.add_callback(DiskSpaceCallback(
    config.OUTPUT_DIR, 
    model, 
    tokenizer, 
    DEMO_SAMPLES
))

# Train
train_result = trainer.train()

# Save final model
print("\n💾 Saving final model...")
trainer.save_model(config.OUTPUT_DIR)
tokenizer.save_pretrained(config.OUTPUT_DIR)

# Log training results
print("\n📊 Training Results:")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")

# ============================================================================
# CELL 10: EVALUATE ON FULL DATASET
# ============================================================================

print("\n" + "="*60)
print("📈 FINAL EVALUATION")
print("="*60)

# Evaluate
eval_results = trainer.evaluate()

print(f"\n📊 Final Metrics:")
print(f"   BLEU Score: {eval_results.get('eval_bleu', 'N/A'):.2f}")
print(f"   chrF++ Score: {eval_results.get('eval_chrf', 'N/A'):.2f}")
print(f"   Eval Loss: {eval_results.get('eval_loss', 'N/A'):.4f}")

# ============================================================================
# CELL 11: SAVE FOR SUBMISSION
# ============================================================================

import shutil

# Create submission package
SUBMISSION_DIR = "./submission_model"
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Copy model files
for file in os.listdir(config.OUTPUT_DIR):
    src = os.path.join(config.OUTPUT_DIR, file)
    dst = os.path.join(SUBMISSION_DIR, file)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print(f"\n✅ Model saved to {SUBMISSION_DIR}")
print("   Files:", os.listdir(SUBMISSION_DIR))

# ============================================================================
# CELL 12: QUICK INFERENCE TEST
# ============================================================================

print("\n" + "="*60)
print("🧪 INFERENCE TEST")
print("="*60)

# Load the trained model
model.eval()
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Test samples
test_samples = [
    "KIŠIB ma-nu-ba-lúm DUMU e-na-sú-en",
    "1 ma-na KÙ.BABBAR ṣa-ru-pá-am",
    "um-ma šu-ma a-na pù-šu-ki-in qí-bi-ma",
]

print("\nTest Translations:")
for sample in test_samples:
    input_text = preprocess_input(sample)
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.MAX_TARGET_LENGTH,
            num_beams=config.NUM_BEAMS,
            length_penalty=config.LENGTH_PENALTY,
            early_stopping=True,
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    decoded = postprocess_output(decoded)
    
    print(f"\n  Input:  {sample}")
    print(f"  Output: {decoded}")

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"""
Next Steps:
1. Download model from '{SUBMISSION_DIR}'
2. Use in your inference notebook
3. Submit to Kaggle

Expected LB: ~33-35 (matching teacher ensemble)
""")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
STUDENT MODEL TRAINING
PyTorch: 2.8.0+cu126
CUDA Available: True
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB

📋 Configuration:
   Model: google/byt5-base
   Epochs: 15
   Batch Size: 2 x 8 = 16
   Learning Rate: 5e-05
   FP16: False, BF16: False
   Use Teacher Translations: True
✓ Preprocessing functions defined

📂 Loading data...
   Loaded teacher translations: 1561 rows
   Using: teacher_translation as target
   After cleaning: 1561 rows

📝 Sample data:
   Input:  translate Akkadian to English: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB š...
   Target: Seal of Mannum-balum-Aššur son of Ṣill-Adad, seal of Šu-Illil son of Mannum-kī-A...

🔤 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

   Train: 1404, Eval: 157
   Tokenizing datasets...


...

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



💾 Saving final model...

📊 Training Results:
   Total steps: 1320
   Training loss: 0.6800

📈 FINAL EVALUATION



────────────────────────────────────────────────────────────
📝 DEMO TRANSLATIONS (Step 1320, BLEU: 32.17)
────────────────────────────────────────────────────────────

[1] Input:  um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni...
    Output: From the colony Kanesh to Aqil<big_gap> the datum, your instructions and wabarra...

[2] Input:  i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ...
    Output: On the tablet that I gave to the City when the week of the City when the week of...
────────────────────────────────────────────────────────────


📊 Final Metrics:
   BLEU Score: 32.17
   chrF++ Score: 47.40
   Eval Loss: 0.3376


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



✅ Model saved to ./submission_model
   Files: ['added_tokens.json', 'tokenizer_config.json', 'training_args.bin', 'special_tokens_map.json', 'generation_config.json', 'config.json', 'model.safetensors']

🧪 INFERENCE TEST

Test Translations:

  Input:  KIŠIB ma-nu-ba-lúm DUMU e-na-sú-en
  Output: Seal of Mannum-bālum son of Enna-Suen.

  Input:  1 ma-na KÙ.BABBAR ṣa-ru-pá-am
  Output: 1 mina of refined silver.

  Input:  um-ma šu-ma a-na pù-šu-ki-in qí-bi-ma
  Output: From Pūšu-kēn: If you arrive to Pūšu-kēn:

🎉 TRAINING COMPLETE!

Next Steps:
1. Download model from './submission_model'
2. Use in your inference notebook
3. Submit to Kaggle

Expected LB: ~33-35 (matching teacher ensemble)

